# C35b — Évaluation du clonage de voix (XTTS-v2, EF-18)

Pour chaque membre de l'équipe qui a donné son accord écrit (ENF-10), XTTS-v2 dit les phrases de
`configs/voice_cloning_eval.yaml` avec sa voix, à partir de ses extraits de référence. Deux
mesures, automatiques :

1. **ressemblance** : cosinus ECAPA (le modèle de C15) entre chaque clone et la voix de référence
   de la personne. Témoins : ses vraies phrases (jamais données à XTTS), et les clones des autres
   membres. La part des clones au-dessus du seuil « inconnu » de C15 (0,5485) dit si notre propre
   identification se laisse tromper ;
2. **intelligibilité** : WER de Whisper small sur les clones, contre le texte demandé (P12).

**Entrées** : un dataset Kaggle **privé**, un dossier par personne (`consentement.txt`,
`reference/*.wav`, `test/*.wav`). Un dossier sans consentement arrête le notebook.
**Sorties** : `c35_evaluation_clonage.json` (chiffres seulement, à copier dans
`training/results/`). Les clones restent dans la sortie Kaggle, privée : ne jamais les publier.

## a. Démarrage

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e "{CLONE_DIR}[ml,training,tts]"

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. Licence, GPU et réglages

XTTS-v2 est sous **Coqui Public Model License** (usage non commercial,
https://coqui.ai/cpml). La personne qui lance le notebook la lit, puis met `ACCEPT_CPML = True`.

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import torch

from echolens.config import CONFIG_DIR
from echolens.core.data.speaker_enrollment import load_speaker_config
from echolens.core.inference import load_inference_config
from echolens.core.models.registry import ModelRegistry, Task
from echolens.training.cloning_eval import load_cloning_eval_config
from echolens.training.transcription_eval import load_transcription_eval_config

ACCEPT_CPML = False
if not ACCEPT_CPML:
    raise RuntimeError("Lire la Coqui Public Model License, puis mettre ACCEPT_CPML = True.")
if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(device)
print("GPU :", GPU_NAME, "· torch", torch.__version__)
NOTEBOOK_START = time.time()

commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
eval_config = load_cloning_eval_config()
inference_config = load_inference_config()
THRESHOLD = inference_config.speaker.threshold
speaker_config = load_speaker_config()
transcription_config = load_transcription_eval_config()
registry = ModelRegistry.from_yaml(CONFIG_DIR / "registry.yaml")
OUTPUT_DIR = Path("/kaggle/working")
CLONES_DIR = OUTPUT_DIR / "c35" / "clones"
CLONES_DIR.mkdir(parents=True, exist_ok=True)
print("Seuil « inconnu » de C15 :", THRESHOLD)

## c. Voix de l'équipe

Le dossier racine est celui qui contient les dossiers de chaque personne, chacun avec son
`consentement.txt`.

In [ ]:
from echolens.training.cloning_eval import find_team_voices

INPUT_DIR = Path("/kaggle/input")
consent_files = sorted(INPUT_DIR.glob(f"**/{eval_config.consent_file}"))
roots = sorted({path.parent.parent for path in consent_files})
if len(roots) != 1:
    raise RuntimeError(f"{len(roots)} dossiers racine de voix trouvés : il en faut exactement un.")
TEAM_DIR = roots[0]
voices = find_team_voices(TEAM_DIR, eval_config)
for voice in voices:
    reference_count = len(voice.reference_paths)
    test_count = len(voice.test_paths)
    print(voice.name, ":", reference_count, "références,", test_count, "tests")

## d. Modèles : XTTS-v2 (registre), ECAPA (C15), Whisper small (C17)

In [ ]:
from echolens.services.xtts_cloner import XttsCloner
from echolens.training.speaker_encoders import load_ecapa
from echolens.training.transcription_models import load_whisper

xtts_entry = registry.for_task(Task.VOICE_CLONING)[0]
cloner = XttsCloner(xtts_entry, inference_config.voice_cloning, ACCEPT_CPML, device="cuda")

ecapa_config = speaker_config.encoders[0]
for encoder_config in speaker_config.encoders:
    if encoder_config.id == eval_config.speaker_encoder:
        ecapa_config = encoder_config
ecapa = load_ecapa(ecapa_config, OUTPUT_DIR / "ecapa", device)

whisper_config = transcription_config.models[0]
for model_config in transcription_config.models:
    if model_config.id == eval_config.transcription_model:
        whisper_config = model_config
whisper = load_whisper(whisper_config, "cuda", "float16", transcription_config)
print("XTTS :", xtts_entry.id, xtts_entry.revision)

## e. Voix de référence, vraies phrases et clones

Pour chaque personne : sa voix de référence (moyenne normée des vecteurs ECAPA de ses extraits,
règle de C15), le score de ses vraies phrases, puis chaque phrase clonée. Le clone évalué est la
sortie de XTTS **avant** le bip de marquage ; la version marquée est gardée pour l'écoute.

In [ ]:
import numpy as np
import soundfile as sf

from echolens.core.audio import load_audio
from echolens.core.speaker_id import voice_print
from echolens.core.synthetic_marker import mark_synthetic
from echolens.training.cloning_eval import cosine_scores
from echolens.training.speaker_encoders import embed_ecapa
from echolens.training.transcription_eval import score_sentence
from echolens.training.transcription_models import transcribe


def ecapa_vectors(paths):
    vectors = []
    for path in paths:
        signal = load_audio(path)
        vectors.append(embed_ecapa(signal, ecapa, device))
    return vectors


voice_prints = {}
real_scores = {}
clone_vectors = {}
transcriptions = {}
synthesis_seconds = []
for voice in voices:
    # Étape 1 : voix de référence et témoin « vraie voix ».
    voice_prints[voice.name] = voice_print(ecapa_vectors(voice.reference_paths))
    test_vectors = ecapa_vectors(voice.test_paths)
    real_scores[voice.name] = cosine_scores(test_vectors, voice_prints[voice.name])
    # Étape 2 : chaque phrase clonée, écrite puis relue en mono 16 kHz (D1, D2).
    folder = CLONES_DIR / voice.name
    folder.mkdir(parents=True, exist_ok=True)
    clone_vectors[voice.name] = []
    transcriptions[voice.name] = []
    for position, sentence in enumerate(eval_config.sentences, start=1):
        start = time.time()
        speech = cloner.synthesize(sentence, voice.reference_paths)
        synthesis_seconds.append(time.time() - start)
        raw_path = folder / f"clone_{position:02d}.wav"
        sf.write(raw_path, speech.samples, speech.sample_rate)
        marked = mark_synthetic(speech.samples, speech.sample_rate, inference_config.voice_cloning)
        sf.write(folder / f"voix_synthetique_{position:02d}.wav", marked, speech.sample_rate)
        clone_signal = load_audio(raw_path)
        clone_vectors[voice.name].append(embed_ecapa(clone_signal, ecapa, device))
        hypothesis = transcribe(whisper, clone_signal, transcription_config)
        transcriptions[voice.name].append(
            score_sentence(str(raw_path), voice.name, sentence, hypothesis)
        )
    print(voice.name, ": vraies phrases", np.round(real_scores[voice.name], 3))

## f. Résultats

Pour chaque personne : clones contre sa voix, ses vraies phrases contre sa voix, clones des
autres membres contre sa voix (s'il y a au moins deux personnes), puis WER.

In [ ]:
import datetime
import shutil

from echolens.training.cloning_eval import voice_report

reports = {}
for voice in voices:
    clone_scores = cosine_scores(clone_vectors[voice.name], voice_prints[voice.name])
    others = []
    for other in voices:
        if other.name != voice.name:
            others.extend(cosine_scores(clone_vectors[other.name], voice_prints[voice.name]))
    reports[voice.name] = voice_report(
        clone_scores, real_scores[voice.name], others, transcriptions[voice.name], THRESHOLD
    )

report = {
    "experiment": "c35_evaluation_clonage",
    "date": datetime.datetime.now(datetime.UTC).isoformat(),
    "branch": BRANCH,
    "commit": commit,
    "gpu": GPU_NAME,
    "model": {"id": xtts_entry.id, "revision": xtts_entry.revision},
    "speaker_encoder": eval_config.speaker_encoder,
    "transcription_model": eval_config.transcription_model,
    "threshold": THRESHOLD,
    "sentence_count": len(eval_config.sentences),
    "voices": reports,
    "mean_synthesis_seconds": float(np.mean(synthesis_seconds)),
    "notebook_minutes": (time.time() - NOTEBOOK_START) / 60,
}
report_path = OUTPUT_DIR / "c35_evaluation_clonage.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
for name, item in reports.items():
    clones = item["clones"]
    real = item["real_sentences"]
    wer = item["transcription"]["wer"]
    print(
        f"{name} : clones {clones['mean']:.3f} ({clones['share_above_threshold']:.0%} au-dessus "
        f"du seuil), vraies phrases {real['mean']:.3f}, WER des clones {wer:.3f}"
    )
shutil.rmtree(CLONE_DIR, ignore_errors=True)